# Hallmark GSEA NES scatter: Olaparib vs 5-FU

Plots `NES_5FU` (x) against `NES_Ola` (y) so the standard reading is:

| region | meaning |
|---|---|
| upper-right (on diagonal) | shared **UP** |
| lower-left (on diagonal) | shared **DOWN** |
| above diagonal, upper-right | olaparib-specific UP |
| below diagonal, upper-right | 5-FU-specific UP |
| upper-left | sign-inverted, olaparib UP / 5-FU DOWN |
| lower-right | sign-inverted, 5-FU UP / olaparib DOWN |

Colour encodes FDR status independently in the two contrasts (4 buckets). Non-significant-in-both terms drop to neutral grey, low alpha.

Inputs are the two HallmarkHuman2025 prerank GSEA result files. The file→drug assignment below was confirmed by correlating each file's leading-edge gene logFCs against the `Control_vs_Ola` and `Control_vs_5-FU` DGE tables (each file correlates maximally with its own contrast).

**Journal-ready output** (matching the EdU figure conventions): single-column (3.5 in), Arial 6 pt, seaborn despine, vector PDF + SVG with editable text and 300-dpi PNG, written to a separate `figures/` folder.

In [ ]:
# --- repo path bootstrap (added by the port) ---
import sys, pathlib
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "utils" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))
from utils.paths import (analysis_input, profiles, features, feature_output, figdir, metadata,
                         data_dir, external, require)
from utils.panels import save_panel

from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.lines import Line2D

try:
    from adjustText import adjust_text
    HAVE_ADJUST = True
except Exception:
    HAVE_ADJUST = False

sns.set_style('white')
sns.set_context('notebook')
%matplotlib inline

In [ ]:
# --- journal-ready rcParams (matches EdU/EdU_analysis.ipynb panels) ----------
# fonttype 42 -> TrueType, so text stays selectable/editable in PDF/EPS (most
# journals require this); svg.fonttype 'none' keeps SVG text as text, not paths.
GRAY = '#9a9a9a'   # axis-line colour, per the Figure 4 design guideline
INK  = '#222222'   # text colour (labels stay dark, axis lines stay gray)

mpl.rcParams.update({
    'font.family':        'sans-serif',
    'font.sans-serif':    ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size':          6,
    'axes.labelsize':     6,
    'axes.titlesize':     6.5,
    'xtick.labelsize':    6,
    'ytick.labelsize':    6,
    'legend.fontsize':    5,
    'axes.linewidth':     0.5,
    'xtick.major.width':  0.5,
    'ytick.major.width':  0.5,
    'xtick.major.size':   2,
    'ytick.major.size':   2,
    'lines.linewidth':    0.8,
    'axes.edgecolor':     GRAY,   # gray axis spines
    'xtick.color':        GRAY,   # gray tick marks
    'ytick.color':        GRAY,
    'xtick.labelcolor':   INK,    # but keep tick labels dark
    'ytick.labelcolor':   INK,
    'axes.labelcolor':    INK,
    'pdf.fonttype':       42,
    'ps.fonttype':        42,
    'svg.fonttype':       'none',
    'figure.facecolor':   'white',
})

In [ ]:
# --- config -----------------------------------------------------------------
# File -> drug assignment confirmed via leading-edge logFC correlation:
#   8d1de180...  best-matches Control_vs_Ola  -> Olaparib
#   b21ff6ac...  best-matches Control_vs_5-FU -> 5-FU
DEG_DATA = analysis_input('3_Figure6/DEG/data')
OLA_GSEA = DEG_DATA / '8d1de180-439d-4f9c-80dd-b2c01c923722_HallmarkHuman2025_gsea_functional_enrichment.csv'
FU_GSEA  = DEG_DATA / 'b21ff6ac-a04d-4c3f-b867-bedbeb1f2f15_HallmarkHuman2025_gsea_functional_enrichment.csv'

CELL_LINE = 'HCT116'           # shown in the title
FIG_W     = 3.5                # single-column width (inches)
FIG_DIR   = figdir('Fig6')    # figures saved here, in a separate folder
OUT_NAME  = 'hallmark_nes_scatter'
FDR_THR   = 0.10              # FDR threshold for the 4-bucket colour encoding
USE_ADJUST = True            # set False to use static label offsets

# --- compact dumbbell geometry (shared by every gene-level dumbbell) ---------
# One set of constants so each dumbbell has the SAME per-gene row height (equal
# y-spacing) and the SAME x-scale, at ~half the old footprint ("twice as small").
DB_WIDTH  = 2.6     # figure width (in) — was 3.5
DB_ROW_H  = 0.13    # inches per gene row -> identical y-spacing across all panels
DB_MARKER = 14      # dot size (was 22)
DB_GENEFS = 4.5     # gene-label font size (was 5)

FIG_DIR.mkdir(exist_ok=True)

In [ ]:
# Hallmark terms to label. Kept to ~15 storytelling points so labels stay legible.
LABEL_TERMS = {
    # shared UP, on diagonal
    'P53 Pathway',
    'Apoptosis',
    # shared DOWN, on diagonal
    'G2M Checkpoint',
    'Mitotic Spindle',
    # olaparib-specific UP (above diagonal, upper-right)
    'Oxidative Phosphorylation',
    # 5-FU-specific UP (below diagonal, upper-right)
    'TNFA Signaling VIA NFKB',
    'Hypoxia',
    'Epithelial Mesenchymal Transition',
    'Coagulation',
    'Reactive Oxygen Species Pathway',
    # sign-inverted, olaparib UP / 5-FU DOWN (upper-left)
    'DNA Repair',
    'Hedgehog Signaling',
    # sign-inverted, 5-FU UP / olaparib DOWN (lower-right)
    'KRAS Signaling UP',
    'UV Response DN',
    # shared DOWN with 5-FU pulling harder
    'E2F Targets',
    'Spermatogenesis',
}

# Drug colour scheme — purple / green (distinct hue families for the two drugs).
COLOURS = {
    'both': '#222222',   # black — the published panel draws 'sig in both' black,
                         # not the burnt orange this had
    'ola':  '#5E3C99',   # royal purple (Olaparib)
    'fu':   '#1B9E77',   # emerald green (5-FU)
    'ns':   '#b8b8b8',   # neutral grey
}
LABEL_COLOUR_OVERRIDE = '#222222'


def short_label(term):
    """Trim a few long Hallmark names so labels don't overrun the figure."""
    table = {
        'Epithelial Mesenchymal Transition': 'EMT',
        'TNFA Signaling VIA NFKB':           'TNFa via NFkB',
        'Reactive Oxygen Species Pathway':   'ROS Pathway',
        'Oxidative Phosphorylation':         'OXPHOS',
        'KRAS Signaling UP':                 'KRAS UP',
        'Hedgehog Signaling':                'Hedgehog',
    }
    return table.get(term, term)


def classify(row, fdr_thr):
    sig_ola = row['FDR_Ola'] < fdr_thr
    sig_fu  = row['FDR_5FU'] < fdr_thr
    if sig_ola and sig_fu:
        return 'both'
    if sig_ola:
        return 'ola'
    if sig_fu:
        return 'fu'
    return 'ns'

## 1. Load both GSEA tables and merge on TERM

In [ ]:
ola = (pd.read_csv(OLA_GSEA)[['TERM', 'NES', 'FDR_Q_VAL']]
       .rename(columns={'NES': 'NES_Ola', 'FDR_Q_VAL': 'FDR_Ola'}))
fu  = (pd.read_csv(FU_GSEA)[['TERM', 'NES', 'FDR_Q_VAL']]
       .rename(columns={'NES': 'NES_5FU', 'FDR_Q_VAL': 'FDR_5FU'}))

df = ola.merge(fu, on='TERM', how='inner')
df['status'] = df.apply(classify, axis=1, fdr_thr=FDR_THR)

print(f'Hallmark terms : {len(df)}')
print(df['status'].value_counts().to_string())
df.head()

## 2. Publication figure

Single-column NES–NES scatter. Point colour = FDR significance bucket; dashed diagonal is `y = x` (shared direction & magnitude); ~16 storytelling hallmarks labelled with `adjustText` repulsion.

In [ ]:
lim = float(np.ceil(max(df['NES_5FU'].abs().max(),
                        df['NES_Ola'].abs().max()) * 10) / 10 + 0.2)

fig, ax = plt.subplots(figsize=(FIG_W, FIG_W))
ax.set_aspect('equal')

# faint crosshair through 0 (quadrant divider) + diagonal y = x
ax.axhline(0, color='#cfcfcf', lw=0.5, zorder=1)
ax.axvline(0, color='#cfcfcf', lw=0.5, zorder=1)
ax.plot([-lim, lim], [-lim, lim], color='#555555', lw=0.7, ls='--', zorder=1)

# background (NS in both) first, then significant on top. No marker edge.
for key in ['ns', 'fu', 'ola', 'both']:
    sub = df[df['status'] == key]
    if sub.empty:
        continue
    ax.scatter(
        sub['NES_5FU'], sub['NES_Ola'],
        s=26 if key != 'ns' else 14,
        c=COLOURS[key],
        alpha=0.35 if key == 'ns' else 0.95,
        edgecolor='none',
        zorder=3 if key != 'ns' else 2,
    )

# labels (repelled, no pointer/leader lines)
texts = []
for _, row in df.iterrows():
    if row['TERM'] not in LABEL_TERMS:
        continue
    texts.append(
        ax.text(
            row['NES_5FU'], row['NES_Ola'],
            short_label(row['TERM']),
            fontsize=5, color=LABEL_COLOUR_OVERRIDE,
            ha='left', va='bottom', zorder=5,
        )
    )

if HAVE_ADJUST and USE_ADJUST:
    adjust_text(
        texts,
        ax=ax,
        expand_points=(1.3, 1.5),
        expand_text=(1.2, 1.3),
        force_text=(0.6, 0.8),
        force_points=(0.3, 0.5),
        only_move={'points': 'xy', 'text': 'xy'},
    )

# quadrant annotations
ann_kwargs = dict(fontsize=5, color='#666666', ha='center', va='center', style='italic')
ax.text( lim * 0.60,  lim * 0.95, 'shared UP',       **ann_kwargs)
ax.text(-lim * 0.60, -lim * 0.95, 'shared DOWN',     **ann_kwargs)
ax.text(-lim * 0.60,  lim * 0.95, 'Ola UP, 5-FU DN', **ann_kwargs)
ax.text( lim * 0.60, -lim * 0.95, '5-FU UP, Ola DN', **ann_kwargs)

# legend
legend_elems = [
    Line2D([0], [0], marker='o', linestyle='', markerfacecolor=COLOURS['both'],
           markeredgecolor='none', markersize=4.5, label=f'sig in both (FDR<{FDR_THR:g})'),
    Line2D([0], [0], marker='o', linestyle='', markerfacecolor=COLOURS['ola'],
           markeredgecolor='none', markersize=4.5, label='sig in Olaparib only'),
    Line2D([0], [0], marker='o', linestyle='', markerfacecolor=COLOURS['fu'],
           markeredgecolor='none', markersize=4.5, label='sig in 5-FU only'),
    Line2D([0], [0], marker='o', linestyle='', markerfacecolor=COLOURS['ns'],
           markeredgecolor='none', markersize=4, alpha=0.5, label='NS in both'),
    Line2D([0], [0], linestyle='--', color='#555555', lw=0.7, label='y = x'),
]
ax.legend(handles=legend_elems, loc='lower right',
          frameon=True, framealpha=0.95, borderpad=0.5,
          handletextpad=0.4, labelspacing=0.3).get_frame().set_linewidth(0.5)

ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_xlabel('Hallmark NES, 5-FU vs DMSO')
ax.set_ylabel('Hallmark NES, Olaparib vs DMSO')
ax.set_title(f'Hallmark GSEA — {CELL_LINE} spheroids', pad=4)
ax.set_xticks(np.arange(-3, 4, 1))
ax.set_yticks(np.arange(-3, 4, 1))

# gray L-axes: left + bottom spines meet at the bottom-left (-3, -3) corner
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(direction='out')

# footnote (method + n), in the EdU panel style
footnote = f'Hallmark MSigDB 2025 · prerank GSEA · {len(df)} gene sets · colour = FDR<{FDR_THR:g}'
fig.text(0.5, -0.02, footnote, ha='center', va='top', fontsize=5, color='#444444')

fig.tight_layout()

# save vector (PDF + SVG, editable text) and 300-dpi PNG into figures/
save_panel(fig, 'Fig6c', data=df,
           caption='Hallmark GSEA NES, olaparib vs 5-FU',
           notebook='analysis/3_Figure6/DEG/hallmark_nes_scatter.ipynb')
plt.show()
plt.close()

if not HAVE_ADJUST:
    print('note: adjustText not installed -- labels placed without repulsion.')
print(f'Saved → {FIG_DIR / OUT_NAME}.pdf / .svg / .png')

## 3. Load per-gene DGE tables

Load the Control-vs-drug differential-expression tables and build `ola_m` / `fu_m` (gene → logFC, FDR), used by every gene-level panel below.

In [ ]:
# --- load per-gene DGE tables -----------------------------------------------
from matplotlib.colors import to_rgba

OLA_DGE = DEG_DATA / 'QMMFHL-colo52_Control_vs_colo52_Ola-dge.csv'
FU_DGE  = DEG_DATA / 'QMMFHL-colo52_Control_vs_colo52_5-FU-dge.csv'


def gene_map(path):
    """gene_name -> (logFC, FDR), de-duplicated to the first occurrence."""
    d = pd.read_csv(path)[['gene_name', 'logFC', 'FDR']].dropna(subset=['gene_name'])
    return d.drop_duplicates('gene_name').set_index('gene_name')


ola_m, fu_m = gene_map(OLA_DGE), gene_map(FU_DGE)
print(f'per-gene DGE loaded — genes (Ola / 5-FU): {len(ola_m)} / {len(fu_m)}')

## 4. OXPHOS gene-level logFC dumbbell

`Oxidative Phosphorylation` is the olaparib-specific UP hallmark in the NES scatter (above the diagonal, upper-right). This companion drills into the electron-transport-chain subunits behind that signal, grouped by respiratory complex (I → V), in the gene-level dumbbell style: **Olaparib (blue)** vs **5-FU (orange)** log₂ fold-change vs DMSO, solid = FDR < 0.10, faded = NS.

In [ ]:
# --- OXPHOS companion: ETC subunits, grouped by respiratory complex ----------
# Canonical electron-transport-chain subunits (HGNC symbols), grouped by the
# complex they assemble into. All verified present in both DGE tables.
OXPHOS_SETS = {
    'Complex I (NADH dehydr.)': [
        'NDUFA9', 'NDUFB8', 'NDUFS1', 'NDUFS2', 'NDUFS3', 'NDUFV1', 'NDUFV2',
    ],
    'Complex II (SDH)': [
        'SDHA', 'SDHB', 'SDHC', 'SDHD',
    ],
    'Complex III (cyt bc1)': [
        'UQCRC1', 'UQCRC2', 'CYC1', 'UQCRFS1', 'UQCRB',
    ],
    'Complex IV (COX)': [
        'COX4I1', 'COX5A', 'COX5B', 'COX6C', 'COX7C', 'CYCS',
    ],
    'Complex V (ATP synthase)': [
        'ATP5F1A', 'ATP5F1B', 'ATP5F1C', 'ATP5PO', 'ATP5MC1',
    ],
}

# tidy table of the OXPHOS genes (reuses ola_m / fu_m gene maps from section 4)
ox_records, ox_missing = [], []
for cat, genes in OXPHOS_SETS.items():
    for g in genes:
        if g in ola_m.index and g in fu_m.index:
            ox_records.append(dict(
                category=cat, gene=g,
                lfc_ola=ola_m.loc[g, 'logFC'], fdr_ola=ola_m.loc[g, 'FDR'],
                lfc_fu=fu_m.loc[g, 'logFC'],   fdr_fu=fu_m.loc[g, 'FDR'],
            ))
        else:
            ox_missing.append(g)
oxdf = pd.DataFrame(ox_records)
if ox_missing:
    print('dropped (not found in both DGE tables):', ox_missing)

# ---- build the dumbbell (compact geometry, same row height as §5) -----------
ox_cats   = list(OXPHOS_SETS.keys())
ox_counts = [int((oxdf['category'] == c).sum()) for c in ox_cats]
ox_xmax   = float(np.ceil(oxdf[['lfc_ola', 'lfc_fu']].abs().max().max() * 2) / 2 + 0.2)

ox_fig_h = DB_ROW_H * sum(ox_counts) + 0.30 * len(ox_cats) + 0.8
fig, axes = plt.subplots(
    len(ox_cats), 1, figsize=(DB_WIDTH, ox_fig_h), sharex=True,
    gridspec_kw=dict(height_ratios=ox_counts, hspace=0.5),
)

for ax, cat in zip(axes, ox_cats):
    sub = (oxdf[oxdf['category'] == cat]
           .sort_values('lfc_fu')          # most-down at bottom, most-up at top
           .reset_index(drop=True))
    y = np.arange(len(sub))

    ax.axvline(0, color=GRAY, lw=0.5, ls='--', zorder=0)
    ax.hlines(y, sub['lfc_ola'], sub['lfc_fu'], color='#cccccc', lw=0.8, zorder=1)

    # opacity encodes significance (solid = FDR<thr, faded = NS); no marker edge
    c_ola = [to_rgba(COLOURS['ola'], 1.0 if s else 0.30) for s in sub['fdr_ola'] < FDR_THR]
    c_fu  = [to_rgba(COLOURS['fu'],  1.0 if s else 0.30) for s in sub['fdr_fu']  < FDR_THR]
    ax.scatter(sub['lfc_ola'], y, s=DB_MARKER, c=c_ola, edgecolor='none', zorder=3)
    ax.scatter(sub['lfc_fu'],  y, s=DB_MARKER, c=c_fu,  edgecolor='none', zorder=3)

    ax.set_yticks(y)
    ax.set_yticklabels(sub['gene'], fontsize=DB_GENEFS)
    ax.set_ylim(-0.6, len(sub) - 0.4)
    ax.set_xlim(-ox_xmax, ox_xmax)
    ax.set_title(cat, fontsize=5.5, loc='left', pad=2)
    ax.tick_params(left=False)
    sns.despine(ax=ax, left=True)

axes[-1].set_xlabel('log$_2$ fold-change vs DMSO')

leg = [
    Line2D([0], [0], marker='o', linestyle='', markerfacecolor=COLOURS['ola'],
           markeredgecolor='none', markersize=4.5, label='Olaparib'),
    Line2D([0], [0], marker='o', linestyle='', markerfacecolor=COLOURS['fu'],
           markeredgecolor='none', markersize=4.5, label='5-FU'),
]
axes[0].legend(handles=leg, loc='lower right', frameon=False,
               fontsize=5, handletextpad=0.3, labelspacing=0.3)

fig.suptitle(f'OXPHOS gene-level log$_2$FC — {CELL_LINE} spheroids', fontsize=6.5, y=0.998)
footnote = f'solid = FDR<{FDR_THR:g}, faded = NS · dot = log$_2$FC vs DMSO · line = cross-drug difference'
fig.text(0.5, 0.004, footnote, ha='center', va='top', fontsize=4.5, color='#444444')

fig.tight_layout(rect=(0, 0.015, 1, 0.985))

OUT4 = 'oxphos_logfc_dumbbell'
# [not a paper panel] fig.savefig(FIG_DIR / f'{OUT4}.pdf', bbox_inches='tight')
# [not a paper panel] fig.savefig(FIG_DIR / f'{OUT4}.svg', bbox_inches='tight')
# [not a paper panel] fig.savefig(FIG_DIR / f'{OUT4}.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()
print(f'Saved → {FIG_DIR / OUT4}.pdf / .svg / .png')

## 5. Combined gene-set signature dumbbells

All five gene-set signatures in **one figure** (built in §7, once the cell-cycle lists are defined), so the panels are directly comparable: every panel shares **one x-axis scale** (same log₂FC spacing) and an **identical per-gene row height** (`height_ratios` ∝ gene count → equal y-spacing everywhere). Each panel carries its **prerank-GSEA NES for both drugs** inside the axes (Olaparib blue / 5-FU orange).

Panels (top 15 genes by |5-FU log₂FC|; **Olaparib (blue)** vs **5-FU (orange)** log₂FC vs DMSO, solid = FDR < 0.10, faded = NS):

- **p53 pathway** → `HALLMARK_P53_PATHWAY` (M5933) — the driver (strongest single signal). Top movers: CDKN1A/p21, MDM2, FAS, PLK2, BTG2.
- **p53 apoptosis** → `REACTOME_TP53_REGULATES_TRANSCRIPTION_OF_CELL_DEATH_GENES` (M27499, C2:CP:REACTOME).
- **E2F targets** → `HALLMARK_E2F_TARGETS` (M5925) and **G2M checkpoint** → `HALLMARK_G2M_CHECKPOINT` (M5901) — the strongest **shared DOWN** terms; gene-level confirmation that 5-FU shuts down the cell cycle (CDKN1A/p21 the lone up-going inhibitor).
- **SASP** → `SAUL_SEN_MAYO` (M45803, C2:CGP) — gold-standard secreted-SASP panel.

Gene-set *membership* is defined across the next two cells (REACTOME/SenMayo here, p53-pathway via GMT in §6, E2F/G2M in §7); the figure itself is rendered by the final call in §7.

*An EMT panel was intentionally dropped:* `HOLLERN_EMT_BREAST_TUMOR_UP` is a bulk breast-tumour signature laden with immune/stromal genes meaningless in a pure HCT116 spheroid, and the direct cadherin/keratin markers show epithelial-up / mesenchymal-down — i.e. no EMT.

In [ ]:
# --- published MSigDB gene sets (full membership) ----------------------------
# Each list is the set's membership as defined in MSigDB (systematic name noted).

REACTOME_TP53_CELL_DEATH = ['AIFM2','APAF1','ATM','BAX','BBC3','BCL2L14','BCL6','BID','BIRC5','BNIP3L','CASP1','CASP10','CASP2','CASP6','CRADD','CREBBP','FAS','IGFBP3','NDRG1','NLRC4','PERP','PIDD1','PMAIP1','PPP1R13B','PRELID1','PRELID3A','RABGGTA','RABGGTB','STEAP3','TMEM219','TNFRSF10A','TNFRSF10B','TNFRSF10C','TNFRSF10D','TP53','TP53AIP1','TP53BP2','TP53I3','TP53INP1','TP63','TP73','TRIAP1','ZNF420']                                                                              # M27499, C2:CP:REACTOME (TP53 regulates transcription of cell death genes)

SAUL_SEN_MAYO = ['ACVR1B','ANG','ANGPT1','ANGPTL4','AREG','AXL','BEX3','BMP2','BMP6','C3','CCL1','CCL13','CCL16','CCL2','CCL20','CCL24','CCL26','CCL3','CCL4','CCL5','CCL7','CCL8','CD55','CD9','CSF1','CSF2','CSF2RB','CST4','CTNNB1','CTSB','CXCL1','CXCL10','CXCL12','CXCL16','CXCL2','CXCL3','CXCL8','CXCR2','DKK1','EDN1','EGF','EGFR','EREG','ESM1','ETS2','FAS','FGF1','FGF2','FGF7','GDF15','GEM','GMFG','HGF','HMGB1','ICAM1','ICAM3','IGF1','IGFBP1','IGFBP2','IGFBP3','IGFBP4','IGFBP5','IGFBP6','IGFBP7','IL10','IL13','IL15','IL18','IL1A','IL1B','IL2','IL32','IL6','IL6ST','IL7','INHA','IQGAP2','ITGA2','ITPKA','JUN','KITLG','LCP1','MIF','MMP1','MMP10','MMP12','MMP13','MMP14','MMP2','MMP3','MMP9','NAP1L4','NRG1','PAPPA','PECAM1','PGF','PLAT','PLAU','PLAUR','PTBP1','PTGER2','PTGES','RPS6KA5','SCAMP4','SELPLG','SEMA3F','SERPINB4','SERPINE1','SERPINE2','SPP1','SPX','TIMP2','TNF','TNFRSF10C','TNFRSF11B','TNFRSF1A','TNFRSF1B','TUBGCP2','VEGFA','VEGFC','VGF','WNT16','WNT2']   # M45803, C2:CGP (SenMayo)


def _gsea_nes_lookup(path=str(analysis_input('3_Figure6/DEG/data/gsea_prerank_results.csv'))):
    """MSigDB term -> {'Ola': (NES, FDR), '5FU': (NES, FDR)} from the cached
    prerank GSEA results (same file the §8 overview uses)."""
    g = pd.read_csv(path)
    g.columns = [c.strip() for c in g.columns]
    fdr_col = next(c for c in g.columns if 'FDR' in c)
    out = {}
    for _, r in g.iterrows():
        out.setdefault(r['Term'], {})[r['drug']] = (float(r['NES']), float(r[fdr_col]))
    return out


def signature_dumbbell(panels, suptitle, outname, topn=15,
                       width=DB_WIDTH, row_h=DB_ROW_H):
    """One combined multi-panel log2FC dumbbell across gene-set signatures.

    panels : {panel_label: (gene_list, gsea_term)} — gsea_term keys the NES text.

    All panels share ONE x-axis (identical log2FC scale everywhere) and an
    identical per-gene row height (height_ratios proportional to gene count ->
    equal y-spacing in every panel). Each panel is annotated, inside the axes,
    with its prerank-GSEA NES for both drugs (Olaparib blue / 5-FU orange).
    Top `topn` genes per panel by |5-FU log2FC|. Reuses ola_m / fu_m / COLOURS /
    FDR_THR from earlier cells.
    """
    nes_lookup = _gsea_nes_lookup()

    recs, counts_seen = [], {}
    for lab, (genes, _term) in panels.items():
        present = [g for g in genes if g in ola_m.index and g in fu_m.index]
        counts_seen[lab] = (len(present), len(genes))
        order = fu_m.reindex(present)['logFC'].abs().sort_values(ascending=False).head(topn).index
        for g in order:
            recs.append(dict(panel=lab, gene=g,
                             lfc_ola=ola_m.loc[g, 'logFC'], fdr_ola=ola_m.loc[g, 'FDR'],
                             lfc_fu=fu_m.loc[g, 'logFC'],   fdr_fu=fu_m.loc[g, 'FDR']))
    d = pd.DataFrame(recs)

    labs  = list(panels.keys())
    nrows = [int((d['panel'] == l).sum()) for l in labs]
    xmax  = float(np.ceil(d[['lfc_ola', 'lfc_fu']].abs().max().max() * 2) / 2 + 0.2)  # GLOBAL x-scale

    fig_h = row_h * sum(nrows) + 0.20 * len(labs) + 0.6
    fig, axes = plt.subplots(len(labs), 1, figsize=(width, fig_h), sharex=True,
                             gridspec_kw=dict(height_ratios=nrows, hspace=0.32))
    if len(labs) == 1:
        axes = [axes]

    for ax, lab in zip(axes, labs):
        sub = d[d['panel'] == lab].sort_values('lfc_fu').reset_index(drop=True)
        y = np.arange(len(sub))
        ax.axvline(0, color=GRAY, lw=0.5, ls='--', zorder=0)
        ax.hlines(y, sub['lfc_ola'], sub['lfc_fu'], color='#cccccc', lw=0.8, zorder=1)
        c_ola = [to_rgba(COLOURS['ola'], 1.0 if s else 0.30) for s in sub['fdr_ola'] < FDR_THR]
        c_fu  = [to_rgba(COLOURS['fu'],  1.0 if s else 0.30) for s in sub['fdr_fu']  < FDR_THR]
        ax.scatter(sub['lfc_ola'], y, s=DB_MARKER, c=c_ola, edgecolor='none', zorder=3)
        ax.scatter(sub['lfc_fu'],  y, s=DB_MARKER, c=c_fu,  edgecolor='none', zorder=3)
        ax.set_yticks(y); ax.set_yticklabels(sub['gene'], fontsize=DB_GENEFS)
        ax.set_ylim(-0.6, len(sub) - 0.4)
        ax.set_xlim(-xmax, xmax)                       # identical x-scale on every panel
        npres, ntot = counts_seen[lab]
        ax.set_title(f'{lab}  ({npres}/{ntot} in DGE)', fontsize=5.5, loc='left', pad=2)
        ax.tick_params(left=False); sns.despine(ax=ax, left=True)

        # NES text inside the panel, both drugs colour-coded (upper-left corner)
        nes = nes_lookup.get(panels[lab][1], {})
        def _f(drug):
            v = nes.get(drug)
            return f'{v[0]:+.2f}' if v else 'NA'
        ax.text(0.015, 0.96, f'NES  Ola {_f("Ola")}', transform=ax.transAxes,
                fontsize=5, fontweight='bold', color=COLOURS['ola'], ha='left', va='top',
                bbox=dict(boxstyle='round,pad=0.12', fc='white', ec='none', alpha=0.75))
        ax.text(0.015, 0.80, f'        5-FU {_f("5FU")}', transform=ax.transAxes,
                fontsize=5, fontweight='bold', color=COLOURS['fu'], ha='left', va='top',
                bbox=dict(boxstyle='round,pad=0.12', fc='white', ec='none', alpha=0.75))

    axes[-1].set_xlabel('log$_2$ fold-change vs DMSO')
    leg = [Line2D([0], [0], marker='o', linestyle='', markerfacecolor=COLOURS['ola'],
                  markeredgecolor='none', markersize=4.5, label='Olaparib'),
           Line2D([0], [0], marker='o', linestyle='', markerfacecolor=COLOURS['fu'],
                  markeredgecolor='none', markersize=4.5, label='5-FU')]
    axes[0].legend(handles=leg, loc='lower right', frameon=False,
                   fontsize=5, handletextpad=0.3, labelspacing=0.3)
    fig.suptitle(suptitle, fontsize=6.5, y=0.998)
    fig.text(0.5, 0.004,
             f'top {topn} genes/panel by |5-FU log$_2$FC| · same x-scale & row height · '
             f'NES from prerank GSEA · solid = FDR<{FDR_THR:g}, faded = NS',
             ha='center', va='top', fontsize=4.5, color='#444444')
    fig.tight_layout(rect=(0, 0.015, 1, 0.985))
    save_panel(fig, outname, data=d, caption=suptitle,
               notebook='analysis/3_Figure6/DEG/hallmark_nes_scatter.ipynb')
    plt.show(); plt.close()
    print(f'Saved → {FIG_DIR / outname}.pdf / .svg / .png  '
          + ' | '.join(f'{l}: {counts_seen[l][0]}/{counts_seen[l][1]}' for l in labs))

# The single combined figure is built in §7 (it needs the cell-cycle gene-set
# lists defined there too).

## 6. p53-pathway membership (feeds the combined figure)

Loads `HALLMARK_P53_PATHWAY` membership from the MSigDB GMT (no hardcoded 200-gene list). This is the strongest single signal in the dataset (NES +2.83 in 5-FU, FDR<0.01) — the p53 transcriptional program driving the arrest / apoptosis / senescence response. It becomes the first panel of the combined signature figure (§7).

In [ ]:
# --- HALLMARK_P53_PATHWAY membership (for the combined figure) ---------------
# The 200 symbols are shipped as an analysis input, extracted once from MSigDB
# h.all.v2026.1.Hs.symbols.gmt. Reading the 48 KB GMT out of the upstream tree used
# to be the only route, and it made this notebook -- and therefore Fig 6d and
# Suppl 6b -- fail outright on any machine without that tree.
def _p53_members():
    members = analysis_input('3_Figure6/DEG/data/hallmark_p53_pathway.txt')
    if members.exists():
        return [g.strip() for g in open(members)
                if g.strip() and not g.startswith('#')]
    # Fall back to the GMT where the bulk inputs are available.
    gmt = external('spher_colo52_v1/3_Figure6/DEG/genesets/h.all.v2026.1.Hs.symbols.gmt')
    if gmt.exists():
        for line in open(gmt):
            parts = line.rstrip('\n').split('\t')
            if parts[0] == 'HALLMARK_P53_PATHWAY':
                return parts[2:]
    raise FileNotFoundError(
        f'HALLMARK_P53_PATHWAY membership not found at {members} and no MSigDB GMT '
        f'at {gmt}. Fetch the analysis inputs with scripts/download_data.py.')


HALLMARK_P53_PATHWAY = _p53_members()

## 7. Cell-cycle membership + render the combined figure

Defines the two canonical proliferation gene sets — `HALLMARK_E2F_TARGETS` (M5925) and `HALLMARK_G2M_CHECKPOINT` (M5901), the strongest **shared DOWN** terms in the NES scatter (§2) — then renders the **single combined `signature_dumbbell` figure** over all five signatures (p53 pathway, p53 apoptosis, E2F, G2M, SASP).

Shared x-scale + equal per-gene row height across every panel; each panel annotated with its prerank-GSEA NES for both drugs. Expected read in the cell-cycle panels: nearly everything down under 5-FU, with **CDKN1A (p21)** — the lone cell-cycle *inhibitor* in E2F_TARGETS — going sharply up, i.e. the arrest signal.

In [ ]:
# --- cell-cycle Hallmark gene sets (full membership) -------------------------
HALLMARK_E2F_TARGETS = ['AK2','ANP32E','ASF1A','ASF1B','ATAD2','AURKA','AURKB','BARD1','BIRC5','BRCA1','BRCA2','BRMS1L','BUB1B','CBX5','CCNB2','CCNE1','CCP110','CDC20','CDC25A','CDC25B','CDCA3','CDCA8','CDK1','CDK4','CDKN1A','CDKN1B','CDKN2A','CDKN2C','CDKN3','CENPE','CENPM','CHEK1','CHEK2','CIT','CKS1B','CKS2','CNOT9','CSE1L','CTCF','CTPS1','DCK','DCLRE1B','DCTPP1','DDX39A','DEK','DEPDC1','DIAPH3','DLGAP5','DNMT1','DONSON','DSCC1','DUT','E2F8','EED','EIF2S1','ESPL1','EXOSC8','EZH2','GINS1','GINS3','GINS4','GSPT1','H2AX','H2AZ1','HELLS','HMGA1','HMGB2','HMGB3','HMMR','HNRNPD','HUS1','ILF3','ING3','IPO7','JPT1','KIF18B','KIF22','KIF2C','KIF4A','KPNA2','LBR','LIG1','LMNB1','LUC7L3','LYAR','MAD2L1','MCM2','MCM3','MCM4','MCM5','MCM6','MCM7','MELK','MKI67','MLH1','MMS22L','MRE11','MSH2','MTHFD2','MXD3','MYBL2','MYC','NAA38','NAP1L1','NASP','NBN','NCAPD2','NME1','NOLC1','NOP56','NUDT21','NUP107','NUP153','NUP205','ORC2','ORC6','PA2G4','PAICS','PAN2','PCNA','PDS5B','PHF5A','PLK1','PLK4','PMS2','PNN','POLA2','POLD1','POLD2','POLD3','POLE','POLE4','POP7','PPM1D','PPP1R8','PRDX4','PRIM2','PRKDC','PRPS1','PSIP1','PSMC3IP','PTTG1','RACGAP1','RAD1','RAD21','RAD50','RAD51AP1','RAD51C','RAN','RANBP1','RBBP7','RFC1','RFC2','RFC3','RNASEH2A','RPA1','RPA2','RPA3','RRM2','SHMT1','SLBP','SMC1A','SMC3','SMC4','SMC6','SNRPB','SPAG5','SPC24','SPC25','SRSF1','SRSF2','SSRP1','STAG1','STMN1','SUV39H1','SYNCRIP','TACC3','TBRG4','TCF19','TFRC','TIMELESS','TIPIN','TK1','TMPO','TOP2A','TP53','TRA2B','TRIP13','TUBB','TUBG1','UBE2S','UBE2T','UBR7','UNG','USP1','WDR90','WEE1','XPO1','XRCC6','ZW10']   # M5925, H

HALLMARK_G2M_CHECKPOINT = ['ABL1','AMD1','ARID4A','ATF5','ATRX','AURKA','AURKB','BARD1','BCL3','BIRC5','BRCA2','BUB1','BUB3','CASP8AP2','CBX1','CCNA2','CCNB2','CCND1','CCNF','CCNT1','CDC20','CDC25A','CDC25B','CDC27','CDC45','CDC6','CDC7','CDK1','CDK4','CDKN1B','CDKN2C','CDKN3','CENPA','CENPE','CENPF','CHAF1A','CHEK1','CHMP1A','CKS1B','CKS2','CTCF','CUL1','CUL3','CUL4A','CUL5','DBF4','DDX39A','DKC1','DMD','DR1','DTYMK','E2F1','E2F2','E2F3','E2F4','EFNA5','EGF','ESPL1','EWSR1','EXO1','EZH2','FANCC','FBXO5','FOXN3','G3BP1','GINS2','GSPT1','H2AX','H2AZ1','H2AZ2','HIF1A','HIRA','HMGA1','HMGB3','HMGN2','HMMR','HNRNPD','HNRNPU','HOXC10','HSPA8','HUS1','ILF3','INCENP','JPT1','KATNA1','KIF11','KIF15','KIF20B','KIF22','KIF23','KIF2C','KIF4A','KIF5B','KMT5A','KNL1','KPNA2','KPNB1','LBR','LIG3','LMNB1','MAD2L1','MAP3K20','MAPK14','MARCKS','MCM2','MCM3','MCM5','MCM6','MEIS1','MEIS2','MKI67','MNAT1','MT2A','MTF2','MYBL2','MYC','NASP','NCL','NDC80','NEK2','NOLC1','NOTCH2','NSD2','NUMA1','NUP50','NUP98','NUSAP1','ODC1','ODF2','ORC5','ORC6','PAFAH1B1','PBK','PDS5B','PLK1','PLK4','PML','POLA2','POLE','POLQ','PRC1','PRIM2','PRMT5','PTTG1','PURA','RACGAP1','RAD21','RAD23B','RAD54L','RASAL2','RBL1','RBM14','RPA2','RPS6KA5','SAP30','SFPQ','SLC12A2','SLC38A1','SLC7A1','SLC7A5','SMAD3','SMARCC1','SMC1A','SMC2','SMC4','SNRPD1','SQLE','SRSF1','SRSF10','SRSF2','SS18','STAG1','STIL','STMN1','SUV39H1','SYNCRIP','TACC3','TFDP1','TGFB1','TLE3','TMPO','TNPO2','TOP1','TOP2A','TPX2','TRA2B','TRAIP','TROAP','TTK','UBE2C','UBE2S','UCK2','UPF1','WRN','XPO1','YTHDC1']   # M5901, H


# --- ONE combined figure: all 5 gene-set signatures, shared x-scale & row -----
# height, each panel annotated with its prerank-GSEA NES (both drugs). Each
# value is (gene_list, MSigDB term used to look up the NES).
SIGNATURE_PANELS = {
    'p53 pathway (HALLMARK)':    (HALLMARK_P53_PATHWAY,     'HALLMARK_P53_PATHWAY'),
    'p53 apoptosis (REACTOME)':  (REACTOME_TP53_CELL_DEATH, 'REACTOME_TP53_REGULATES_TRANSCRIPTION_OF_CELL_DEATH_GENES'),
    'E2F targets (HALLMARK)':    (HALLMARK_E2F_TARGETS,     'HALLMARK_E2F_TARGETS'),
    'G2M checkpoint (HALLMARK)': (HALLMARK_G2M_CHECKPOINT,  'HALLMARK_G2M_CHECKPOINT'),
    'SASP (SAUL_SEN_MAYO)':      (SAUL_SEN_MAYO,            'SAUL_SEN_MAYO'),
}

# Fig 6d and Suppl 6b were one combined output upstream; they are separate
# paper panels, so each is drawn from its own subset with its own source table.
FIG6D_KEYS = ['p53 pathway (HALLMARK)', 'E2F targets (HALLMARK)',
              'p53 apoptosis (REACTOME)']            # manuscript order
SUPPL6B_KEYS = ['G2M checkpoint (HALLMARK)', 'SASP (SAUL_SEN_MAYO)']

signature_dumbbell({k: SIGNATURE_PANELS[k] for k in FIG6D_KEYS},
                   suptitle=f'Gene-set signatures — {CELL_LINE} spheroids',
                   outname='Fig6d')

signature_dumbbell({k: SIGNATURE_PANELS[k] for k in SUPPL6B_KEYS},
                   suptitle=f'Gene-set signatures — {CELL_LINE} spheroids',
                   outname='SupplFig6b')

## 8. GSEA NES overview — participation statistic

Mean log₂FC shows direction; **NES** (with FDR, from preranked GSEA on the log₂FC ranking vs MSigDB v2026.1 H + C2) shows whether a set is genuinely *enriched* vs background. This is the **minimal panel — one signature per axis** — for both drugs.

- **Proliferation OFF** (E2F, G2M) and **p53/DDR ON** (p53 pathway, p53 cell-death) are the dominant FDR<0.01 signals; the **5-FU vs Olaparib asymmetry** is explicit (E2F collapses only under 5-FU).
- **OXPHOS** up; **SASP (SenMayo)** only trends (NS) — early/partial senescence + inflammatory-SASP detection floor.
- **EMT** (Kohn epithelial/mesenchymal) is NS in both arms — the honest "no EMT".

Cached to `genesets/gsea_prerank_results.csv`; delete to recompute.

In [ ]:
# --- GSEA NES overview: participation statistic ------------------------------
# NES + FDR from preranked GSEA (logFC ranking) vs MSigDB v2026.1 H + C2.
# Cached to genesets/gsea_prerank_results.csv; delete that file to recompute.
import os

GSEA_CSV = str(analysis_input('3_Figure6/DEG/data/gsea_prerank_results.csv'))
if os.path.exists(GSEA_CSV):
    gsea = pd.read_csv(GSEA_CSV)
else:
    import gseapy as gp
    def _rnk(p):
        d = (pd.read_csv(p)[['gene_name', 'logFC']].dropna()
             .drop_duplicates('gene_name').sort_values('logFC', ascending=False))
        return d.rename(columns={'gene_name': 0, 'logFC': 1})[[0, 1]]
    frames = []
    for coll, gmtf in [('H', str(external('spher_colo52_v1/3_Figure6/DEG/genesets/h.all.v2026.1.Hs.symbols.gmt'))),
                       ('C2', str(external('spher_colo52_v1/3_Figure6/DEG/genesets/c2.all.v2026.1.Hs.symbols.gmt')))]:
        for drug, dge in [('5FU', FU_DGE), ('Ola', OLA_DGE)]:
            pr = gp.prerank(rnk=_rnk(dge), gene_sets=gmtf, min_size=5, max_size=1000,
                            permutation_num=1000, threads=4, seed=1, outdir=None, verbose=False)
            d = pr.res2d.copy(); d['drug'] = drug; d['coll'] = coll; frames.append(d)
    gsea = pd.concat(frames, ignore_index=True)
    gsea.to_csv(GSEA_CSV, index=False)

gsea.columns = [c.strip() for c in gsea.columns]
_FDR = [c for c in gsea.columns if 'FDR' in c][0]

# curated overview panel (label -> MSigDB term)
NES_PANEL = [
    ('p53 pathway',               'HALLMARK_P53_PATHWAY'),
    ('p53 cell-death (Reactome)', 'REACTOME_TP53_REGULATES_TRANSCRIPTION_OF_CELL_DEATH_GENES'),
    ('OXPHOS',                    'HALLMARK_OXIDATIVE_PHOSPHORYLATION'),
    ('SASP (SenMayo)',            'SAUL_SEN_MAYO'),
    ('EMT: epithelial (Kohn)',    'KOHN_EMT_EPITHELIAL'),
    ('EMT: mesenchymal (Kohn)',   'KOHN_EMT_MESENCHYMAL'),
    ('G2M checkpoint',            'HALLMARK_G2M_CHECKPOINT'),
    ('E2F targets',               'HALLMARK_E2F_TARGETS'),
]

def _nes(term, drug):
    r = gsea[(gsea['Term'] == term) & (gsea['drug'] == drug)]
    if not len(r):
        return np.nan, np.nan
    return float(r['NES'].iloc[0]), float(r[_FDR].iloc[0])

nesdf = pd.DataFrame([
    dict(label=lab,
         nes_fu=_nes(t, '5FU')[0], fdr_fu=_nes(t, '5FU')[1],
         nes_ola=_nes(t, 'Ola')[0], fdr_ola=_nes(t, 'Ola')[1])
    for lab, t in NES_PANEL
])

# ---- figure: NES dumbbell (5-FU vs Olaparib), FDR by opacity + stars --------
y = np.arange(len(nesdf))[::-1]
fig, ax = plt.subplots(figsize=(4.4, 0.32 * len(nesdf) + 1.0))
ax.axvline(0, color='#444444', lw=0.6, zorder=1)
for xv in (-2, 2):
    ax.axvline(xv, color='#e3e3e3', lw=0.5, zorder=0)
ax.hlines(y, nesdf['nes_ola'], nesdf['nes_fu'], color='#cccccc', lw=0.8, zorder=2)
c_fu  = [to_rgba(COLOURS['fu'],  1.0 if s else 0.30) for s in nesdf['fdr_fu']  < FDR_THR]
c_ola = [to_rgba(COLOURS['ola'], 1.0 if s else 0.30) for s in nesdf['fdr_ola'] < FDR_THR]
ax.scatter(nesdf['nes_ola'], y, s=26, c=c_ola, edgecolor='none', zorder=3)
ax.scatter(nesdf['nes_fu'],  y, s=30, c=c_fu,  edgecolor='none', zorder=4)

def _star(f):
    return '***' if f < 0.01 else '**' if f < 0.05 else '*' if f < 0.1 else ''
for yi, (_, r) in zip(y, nesdf.iterrows()):
    st = _star(r['fdr_fu'])
    if st:
        off = 0.20 if r['nes_fu'] >= 0 else -0.20
        ax.text(r['nes_fu'] + off, yi, st, fontsize=6, va='center',
                ha='left' if r['nes_fu'] >= 0 else 'right', color='#333333')

ax.set_yticks(y); ax.set_yticklabels(nesdf['label'], fontsize=6)
ax.set_xlim(-3.7, 3.7)
ax.set_xlabel('GSEA NES  (down ← 0 → up)')
ax.tick_params(left=False); sns.despine(ax=ax, left=True)
leg = [Line2D([0], [0], marker='o', ls='', mfc=COLOURS['fu'],  mec='none', ms=5, label='5-FU'),
       Line2D([0], [0], marker='o', ls='', mfc=COLOURS['ola'], mec='none', ms=5, label='Olaparib'),
       Line2D([0], [0], marker='o', ls='', mfc='#bbbbbb',      mec='none', ms=5, label='faded = FDR>=0.1')]
ax.legend(handles=leg, loc='lower right', frameon=False, fontsize=5,
          handletextpad=0.3, labelspacing=0.3)
ax.set_title(f'GSEA participation (NES) - {CELL_LINE} spheroids vs DMSO', fontsize=6.5, pad=4)
fig.text(0.5, -0.01,
         'preranked GSEA (logFC) | MSigDB v2026.1 H + C2 | * FDR<0.1  ** <0.05  *** <0.01',
         ha='center', va='top', fontsize=5, color='#444444')
fig.tight_layout()
for ext in ('pdf', 'svg', 'png'):
        # [not a paper panel] fig.savefig(FIG_DIR / f'gsea_nes_overview.{ext}', dpi=300, bbox_inches='tight')
    pass
plt.show(); plt.close()

# tidy NES table (5-FU and Olaparib)
nes_table = nesdf.rename(columns={'label': 'gene set',
    'nes_fu': '5FU_NES', 'fdr_fu': '5FU_FDR',
    'nes_ola': 'Ola_NES', 'fdr_ola': 'Ola_FDR'}).round(
    {'5FU_NES': 2, '5FU_FDR': 3, 'Ola_NES': 2, 'Ola_FDR': 3})
print(nes_table.to_string(index=False))
nes_table